In [1]:
"""
Problem 3: Find the closest positive semi-definite matrix to C
using Frobenius norm minimization.
"""
 
import numpy as np
from scipy.optimize import minimize
 
# Original (invalid) correlation matrix
C = np.array([
    [1.0,  0.9,  0.9],
    [0.9,  1.0, -0.9],
    [0.9, -0.9,  1.0]
])
 
def vec_to_corr(x):
    """Reconstruct a symmetric matrix with unit diagonal from upper-triangle entries."""
    n = 3
    M = np.eye(n)
    idx = np.triu_indices(n, k=1)
    M[idx] = x
    M[(idx[1], idx[0])] = x
    return M
 
def objective(x):
    """Frobenius norm squared between candidate matrix and C."""
    M = vec_to_corr(x)
    return np.linalg.norm(M - C, 'fro') ** 2
 
def psd_penalty(x, alpha=1e4):
    """Penalty for negative eigenvalues."""
    M = vec_to_corr(x)
    eigvals = np.linalg.eigvalsh(M)
    return alpha * np.sum(np.minimum(eigvals, 0) ** 2)
 
def total_loss(x):
    return objective(x) + psd_penalty(x)
 
# Bounds: correlation entries in [-1, 1]
bounds = [(-1, 1)] * 3
 
# Initial guess: upper-triangle of C
x0 = np.array([C[0, 1], C[0, 2], C[1, 2]])
 
result = minimize(total_loss, x0, method='L-BFGS-B', bounds=bounds,
                  options={'maxiter': 10000, 'ftol': 1e-15, 'gtol': 1e-12})
 
C_psd = vec_to_corr(result.x)
 
# Verify PSD: clip tiny negative eigenvalues numerically
eigvals = np.linalg.eigvalsh(C_psd)
 
print("Original matrix C:")
print(C)
print("\nEigenvalues of C:", np.linalg.eigvalsh(C))
 
print("\nClosest PSD correlation matrix C*:")
print(np.round(C_psd, 6))
print("\nEigenvalues of C*:", np.round(eigvals, 8))
print("\nFrobenius distance ||C* - C||_F:", round(np.linalg.norm(C_psd - C, 'fro'), 6))
print("All eigenvalues >= 0:", np.all(eigvals >= -1e-10))
 
# ── Alternative: analytical nearest PSD via eigenvalue clipping ──────────────
def nearest_psd_clip(A):
    """Project onto PSD cone by zeroing negative eigenvalues (Higham 1988)."""
    eigvals, eigvecs = np.linalg.eigh(A)
    eigvals_clipped = np.maximum(eigvals, 0)
    B = eigvecs @ np.diag(eigvals_clipped) @ eigvecs.T
    # Re-normalise diagonal to 1 (correlation matrix)
    D = np.sqrt(np.diag(B))
    B = B / np.outer(D, D)
    return B
 
C_clip = nearest_psd_clip(C)
print("\n── Analytical nearest PSD (eigenvalue clipping) ──")
print(np.round(C_clip, 6))
print("Eigenvalues:", np.round(np.linalg.eigvalsh(C_clip), 8))
print("Frobenius distance:", round(np.linalg.norm(C_clip - C, 'fro'), 6))

Original matrix C:
[[ 1.   0.9  0.9]
 [ 0.9  1.  -0.9]
 [ 0.9 -0.9  1. ]]

Eigenvalues of C: [-0.8  1.9  1.9]

Closest PSD correlation matrix C*:
[[ 1.        0.497452  0.497452]
 [ 0.497452  1.       -0.497452]
 [ 0.497452 -0.497452  1.      ]]

Eigenvalues of C*: [0.00509553 1.49745223 1.49745224]

Frobenius distance ||C* - C||_F: 0.986037
All eigenvalues >= 0: True

── Analytical nearest PSD (eigenvalue clipping) ──
[[ 1.   0.5  0.5]
 [ 0.5  1.  -0.5]
 [ 0.5 -0.5  1. ]]
Eigenvalues: [-0.   1.5  1.5]
Frobenius distance: 0.979796
